In [4]:
import os
import requests
import urllib3
from datetime import datetime, timedelta

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)


class NPPCoalStockDownloader:

    def __init__(
        self,
        download_folder="coal_stocks_data_test_final"
    ):
        self.download_folder = download_folder

        os.makedirs(
            self.download_folder,
            exist_ok=True
        )

        self.session = requests.Session()

        self.session.headers.update({
            "User-Agent": "Mozilla/5.0",
            "Accept": "*/*"
        })

    def create_urls(self, date):

        day = f"{date.day:02d}"
        month = f"{date.month:02d}"
        year = date.year

        # Primary XLSX
        xlsx_url = (
            "https://npp.gov.in/public-reports/cea/daily/fuel/"
            f"{day}-{month}-{year}/"
            f"dailyCoal1-{year}-{month}-{day}.xlsx"
        )

        # Fallback XLS
        xls_url = (
            "https://npp.gov.in/public-reports/cea/daily/fuel/"
            f"{day}-{month}-{year}/"
            f"dailyCoal1-{year}-{month}-{day}.xls"
        )

        return xlsx_url, xls_url

    def download_file(self, date):

        xlsx_url, xls_url = self.create_urls(date)

        date_string = date.strftime("%Y-%m-%d")

        print(f"\nDownloading: {date_string}")

        # ------------------------------------------------
        # Try XLSX
        # ------------------------------------------------

        print("Trying XLSX:")
        print(xlsx_url)

        try:

            response = self.session.get(
                xlsx_url,
                timeout=(15, 60),
                verify=False
            )

            print(
                "HTTP Status:",
                response.status_code
            )

            if response.status_code == 200:

                filename = (
                    f"dailyCoal1-{date_string}.xlsx"
                )

                filepath = os.path.join(
                    self.download_folder,
                    filename
                )

                with open(filepath, "wb") as f:
                    f.write(response.content)

                print(
                    f"Downloaded XLSX: {filepath}"
                )

                return filepath

        except requests.RequestException as e:

            print(
                f"XLSX request failed: {e}"
            )

        # ------------------------------------------------
        # XLSX failed → Try XLS
        # ------------------------------------------------

        print("\nXLSX unavailable.")
        print("Trying XLS:")
        print(xls_url)

        try:

            response = self.session.get(
                xls_url,
                timeout=(15, 60),
                verify=False
            )

            print(
                "HTTP Status:",
                response.status_code
            )

            if response.status_code == 200:

                filename = (
                    f"dailyCoal1-{date_string}.xls"
                )

                filepath = os.path.join(
                    self.download_folder,
                    filename
                )

                with open(filepath, "wb") as f:
                    f.write(response.content)

                print(
                    f"Downloaded XLS: {filepath}"
                )

                return filepath

        except requests.RequestException as e:

            print(
                f"XLS request failed: {e}"
            )

        # ------------------------------------------------
        # Nothing found
        # ------------------------------------------------

        print(
            f"❌ No report found for {date_string}"
        )

        return None

    def download_date_range(
        self,
        start_date,
        end_date
    ):

        downloaded_files = []
        failed_dates = []

        current_date = start_date

        while current_date <= end_date:

            filepath = self.download_file(
                current_date
            )

            if filepath:

                downloaded_files.append(
                    filepath
                )

            else:

                failed_dates.append(
                    current_date.strftime("%Y-%m-%d")
                )

            current_date += timedelta(days=1)

        # ------------------------------------------------
        # Summary
        # ------------------------------------------------

        print("\n" + "=" * 60)
        print("DOWNLOAD SUMMARY")
        print("=" * 60)

        print(
            f"Downloaded: {len(downloaded_files)}"
        )

        print(
            f"Not found: {len(failed_dates)}"
        )

        if failed_dates:

            print("\nStill unavailable:")

            for date in failed_dates:
                print(date)

        return downloaded_files, failed_dates

In [6]:
from datetime import datetime, timedelta
import yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)
npp_link = config["npp_coal_stocks"]["npp_link"]

folder_location = config["npp_coal_stocks"]["folder_location"]

downloader = NPPCoalStockDownloader(
    
)
end_date = datetime.today() - timedelta(days=1)

start_date = datetime.today() - timedelta(days=28)

downloaded_files = downloader.download_date_range(
    start_date,
    end_date
)

print("\nDownloaded files:")
for file in downloaded_files:
    print(file)


Downloading: 2026-08-12
Trying XLSX:
https://npp.gov.in/public-reports/cea/daily/fuel/12-08-2026/dailyCoal1-2026-08-12.xlsx
HTTP Status: 404

XLSX unavailable.
Trying XLS:
https://npp.gov.in/public-reports/cea/daily/fuel/12-08-2026/dailyCoal1-2026-08-12.xls
HTTP Status: 200
Downloaded XLS: coal_stocks_data_test_final/dailyCoal1-2026-08-12.xls

Downloading: 2026-08-13
Trying XLSX:
https://npp.gov.in/public-reports/cea/daily/fuel/13-08-2026/dailyCoal1-2026-08-13.xlsx
HTTP Status: 404

XLSX unavailable.
Trying XLS:
https://npp.gov.in/public-reports/cea/daily/fuel/13-08-2026/dailyCoal1-2026-08-13.xls
HTTP Status: 200
Downloaded XLS: coal_stocks_data_test_final/dailyCoal1-2026-08-13.xls

Downloading: 2026-08-14
Trying XLSX:
https://npp.gov.in/public-reports/cea/daily/fuel/14-08-2026/dailyCoal1-2026-08-14.xlsx
HTTP Status: 200
Downloaded XLSX: coal_stocks_data_test_final/dailyCoal1-2026-08-14.xlsx

Downloading: 2026-08-15
Trying XLSX:
https://npp.gov.in/public-reports/cea/daily/fuel/15-08-2

In [7]:
import re
import math
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

INPUT_DIR = Path(
    "/Users/shivammishra/Documents/Personal Projects/npp_web/coal_stocks_data_test_final"
)

OUTPUT_CSV = INPUT_DIR / "npp_coal_stocks_final.csv"
AUDIT_CSV = INPUT_DIR / "npp_coal_stocks_audit_final.csv"


OUTPUT_COLUMNS = [
    "Date",
    "Normative Stock Reqd. (Days)",
    "Daily Requirement @85% PLF (In '000 Tonnes)",
    "Actual Stock - Total (In '000 Tonnes)",
]


# ============================================================
# HELPERS
# ============================================================

def norm(value):

    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    text = str(value).lower()

    replacements = {
        "\n": " ",
        "\r": " ",
        "\t": " ",
        "’": "'",
        "‘": "'",
        "–": "-",
        "—": "-",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def is_num(value):

    if value is None:
        return False

    try:
        if pd.isna(value):
            return False
    except Exception:
        pass

    try:
        value = float(value)

        return math.isfinite(value)

    except Exception:
        return False


def num(value):

    if not is_num(value):
        return None

    return float(value)


def clean_num(value):

    if value is None:
        return None

    value = float(value)

    if abs(value - round(value)) < 1e-9:
        return int(round(value))

    return round(value, 6)


def excel_col(col):

    result = ""

    col += 1

    while col:

        col, rem = divmod(
            col - 1,
            26
        )

        result = (
            chr(65 + rem)
            + result
        )

    return result


def extract_date(filename):

    match = re.search(
        r"(20\d{2}-\d{2}-\d{2})",
        filename
    )

    if not match:

        raise ValueError(
            f"Date not found in filename: {filename}"
        )

    return match.group(1)


# ============================================================
# WORKBOOK CLASS
# ============================================================

class WorkbookData:

    def __init__(
        self,
        path,
        sheet_name,
        values
    ):

        self.path = path
        self.sheet_name = sheet_name
        self.values = values

        self.rows = len(values)

        self.cols = (
            max(
                len(row)
                for row in values
            )
            if values
            else 0
        )

    def get(
        self,
        row,
        col
    ):

        if row < 0:
            return None

        if row >= self.rows:
            return None

        if col < 0:
            return None

        if col >= len(
            self.values[row]
        ):
            return None

        return self.values[row][col]

    def row(
        self,
        row
    ):

        if row < 0 or row >= self.rows:
            return []

        return self.values[row]


# ============================================================
# LOAD XLS
# ============================================================

def load_xls(path):

    import xlrd

    wb = xlrd.open_workbook(
        path
    )

    best_sheet = None
    best_score = -1

    for sheet in wb.sheets():

        score = 0

        for r in range(
            min(sheet.nrows, 120)
        ):

            for c in range(
                min(sheet.ncols, 50)
            ):

                value = sheet.cell_value(
                    r,
                    c
                )

                text = norm(
                    value
                )

                if "grand total" in text:
                    score += 100

                if "daily requirement" in text:
                    score += 50

                if "actual stock" in text:
                    score += 50

                if "normative stock" in text:
                    score += 50

        if score > best_score:

            best_score = score
            best_sheet = sheet

    if best_sheet is None:

        raise RuntimeError(
            "DailyCoalReport sheet not found"
        )

    values = []

    for r in range(
        best_sheet.nrows
    ):

        values.append(
            [
                best_sheet.cell_value(
                    r,
                    c
                )
                for c in range(
                    best_sheet.ncols
                )
            ]
        )

    return WorkbookData(
        path,
        best_sheet.name,
        values
    )


# ============================================================
# LOAD XLSX
# ============================================================

def load_xlsx(path):

    from openpyxl import load_workbook

    wb = load_workbook(
        path,
        read_only=True,
        data_only=True
    )

    best_ws = None
    best_score = -1

    for ws in wb.worksheets:

        score = 0

        for row in ws.iter_rows(
            min_row=1,
            max_row=min(
                ws.max_row,
                120
            ),
            max_col=min(
                ws.max_column,
                30
            )
        ):

            for cell in row:

                text = norm(
                    cell.value
                )

                if "grand total" in text:
                    score += 100

                if "daily requirement" in text:
                    score += 50

                if "actual stock" in text:
                    score += 50

                if "normative stock" in text:
                    score += 50

        if score > best_score:

            best_score = score
            best_ws = ws

    if best_ws is None:

        raise RuntimeError(
            "DailyCoalReport sheet not found"
        )

    values = []

    for row in best_ws.iter_rows(
        values_only=True
    ):

        values.append(
            list(row)
        )

    return WorkbookData(
        path,
        best_ws.title,
        values
    )


# ============================================================
# LOAD WORKBOOK
# ============================================================

def load_workbook_data(path):

    if path.suffix.lower() == ".xls":

        return load_xls(path)

    if path.suffix.lower() == ".xlsx":

        return load_xlsx(path)

    raise ValueError(
        f"Unsupported file: {path}"
    )


# ============================================================
# GRAND TOTAL ROW
# ============================================================

def find_grand_total_row(book):

    candidates = []

    for r in range(
        book.rows
    ):

        text = " ".join(
            norm(
                value
            )
            for value in book.row(r)
            if norm(value)
        )

        compact = (
            text
            .replace(" ", "")
            .replace(":", "")
        )

        score = 0

        if "grandtotal" in compact:
            score += 500

        if "a+b+c+d" in compact:
            score += 500

        if "कुलयोग" in compact:
            score += 300

        numeric_count = sum(
            is_num(value)
            for value in book.row(r)
        )

        score += min(
            numeric_count,
            30
        )

        if score > 0:

            candidates.append(
                (
                    score,
                    r
                )
            )

    if not candidates:

        raise RuntimeError(
            "Grand Total row not found"
        )

    candidates.sort(
        key=lambda x: (
            -x[0],
            x[1]
        )
    )

    return candidates[0][1]


# ============================================================
# FIND HEADER ROWS
# ============================================================

def find_header_rows(
    book,
    grand_total_row
):

    rows = []

    for r in range(
        min(
            grand_total_row,
            30
        )
    ):

        text = " ".join(
            norm(
                value
            )
            for value in book.row(r)
        )

        if (
            "normative stock" in text
            or "daily requirement" in text
            or "actual stock" in text
        ):

            rows.append(r)

    return rows


# ============================================================
# DETECT REPORT LAYOUT
# ============================================================

def detect_layout(
    book,
    grand_total_row
):

    suffix = book.path.suffix.lower()

    header_rows = find_header_rows(
        book,
        grand_total_row
    )

    all_header_text = ""

    for r in header_rows:

        all_header_text += " ".join(
            norm(v)
            for v in book.row(r)
        )

    # --------------------------------------------------------
    # XLSX NEW FORMAT
    #
    # Daily = H
    # Normative stock = I
    # Actual total = L
    # --------------------------------------------------------

    if suffix == ".xlsx":

        row_text = " ".join(
            norm(v)
            for v in book.row(
                grand_total_row
            )
        )

        # The new xlsx reports have a very distinctive structure.
        if (
            "daily requirement @85% plf" in all_header_text
            and "normative stock required" in all_header_text
            and "actual stock in" in all_header_text
        ):

            return "XLSX_STANDARD"

    # --------------------------------------------------------
    # XLS FORMAT
    #
    # We distinguish layouts based on where the Grand Total
    # metrics physically occur.
    # --------------------------------------------------------

    if suffix == ".xls":

        row = book.row(
            grand_total_row
        )

        # ----------------------------------------------------
        # Old XLS layout:
        #
        # L = days
        # O = daily requirement
        # R = normative stock
        # Z = actual total
        # ----------------------------------------------------

        if (
            is_num(book.get(
                grand_total_row,
                11
            ))
            and is_num(book.get(
                grand_total_row,
                17
            ))
            and is_num(book.get(
                grand_total_row,
                25
            ))
        ):

            # R should be a large normative stock number.
            normative = num(
                book.get(
                    grand_total_row,
                    17
                )
            )

            days = num(
                book.get(
                    grand_total_row,
                    11
                )
            )

            actual = num(
                book.get(
                    grand_total_row,
                    25
                )
            )

            if (
                normative > 10000
                and 1 < days < 100
                and actual > 0
            ):

                return "XLS_OLD"

        # ----------------------------------------------------
        # New XLS layout:
        #
        # M = days
        # R = normative stock
        # Y = actual total
        # ----------------------------------------------------

        if (
            is_num(book.get(
                grand_total_row,
                12
            ))
            and is_num(book.get(
                grand_total_row,
                17
            ))
            and is_num(book.get(
                grand_total_row,
                24
            ))
        ):

            normative = num(
                book.get(
                    grand_total_row,
                    17
                )
            )

            days = num(
                book.get(
                    grand_total_row,
                    12
                )
            )

            actual = num(
                book.get(
                    grand_total_row,
                    24
                )
            )

            if (
                normative > 10000
                and 1 < days < 100
                and actual > 0
            ):

                return "XLS_NEW"

    raise RuntimeError(
        "Unknown NPP workbook layout"
    )


# ============================================================
# EXTRACT XLSX STANDARD
# ============================================================

def extract_xlsx_standard(
    book,
    grand_total_row
):

    # Excel:
    #
    # H = Daily Requirement
    # I = Normative Stock
    # J = Indigenous
    # K = Import
    # L = Total
    #

    daily_col = 7
    normative_stock_col = 8
    indigenous_col = 9
    import_col = 10
    actual_total_col = 11

    daily = num(
        book.get(
            grand_total_row,
            daily_col
        )
    )

    normative_stock = num(
        book.get(
            grand_total_row,
            normative_stock_col
        )
    )

    indigenous = num(
        book.get(
            grand_total_row,
            indigenous_col
        )
    )

    imported = num(
        book.get(
            grand_total_row,
            import_col
        )
    )

    actual_total = num(
        book.get(
            grand_total_row,
            actual_total_col
        )
    )

    if daily is None:

        raise RuntimeError(
            "XLSX Daily Requirement missing"
        )

    if normative_stock is None:

        raise RuntimeError(
            "XLSX Normative Stock missing"
        )

    if actual_total is None:

        raise RuntimeError(
            "XLSX Actual Stock Total missing"
        )

    # --------------------------------------------------------
    # XLSX reports don't have an explicit Normative Days
    # column.
    #
    # Derive it.
    # --------------------------------------------------------

    normative_days = (
        normative_stock
        / daily
    )

    # --------------------------------------------------------
    # Validate Actual Stock.
    # --------------------------------------------------------

    actual_expected = None
    actual_error = None

    if (
        indigenous is not None
        and imported is not None
    ):

        actual_expected = (
            indigenous
            + imported
        )

        actual_error = (
            abs(
                actual_total
                - actual_expected
            )
            / max(
                abs(actual_expected),
                1
            )
        )

    return {

        "days": normative_days,

        "daily": daily,

        "normative_stock": normative_stock,

        "actual_total": actual_total,

        "daily_col": daily_col,

        "normative_days_col": None,

        "normative_stock_col": normative_stock_col,

        "actual_total_col": actual_total_col,

        "indigenous_col": indigenous_col,

        "import_col": import_col,

        "days_method": "derived",

        "actual_expected": actual_expected,

        "actual_error": actual_error,
    }


# ============================================================
# EXTRACT OLD XLS
# ============================================================

def extract_xls_old(
    book,
    grand_total_row
):

    # --------------------------------------------------------
    # Verified old NPP XLS layout:
    #
    # L = Normative Days
    # O = Daily Requirement
    # R = Normative Stock
    # U/V/W = Indigenous
    # X/Y/Z = Import/Total structure
    #
    # Grand Total:
    #
    # L -> days
    # O -> daily
    # R -> normative stock
    # V -> indigenous
    # X -> import
    # Z -> total
    # --------------------------------------------------------

    days_col = 11
    daily_col = 14
    normative_stock_col = 17
    indigenous_col = 21
    import_col = 23
    actual_total_col = 25

    days = num(
        book.get(
            grand_total_row,
            days_col
        )
    )

    daily = num(
        book.get(
            grand_total_row,
            daily_col
        )
    )

    normative_stock = num(
        book.get(
            grand_total_row,
            normative_stock_col
        )
    )

    indigenous = num(
        book.get(
            grand_total_row,
            indigenous_col
        )
    )

    imported = num(
        book.get(
            grand_total_row,
            import_col
        )
    )

    actual_total = num(
        book.get(
            grand_total_row,
            actual_total_col
        )
    )

    if days is None:

        raise RuntimeError(
            "Old XLS Normative Days missing"
        )

    if normative_stock is None:

        raise RuntimeError(
            "Old XLS Normative Stock missing"
        )

    if actual_total is None:

        raise RuntimeError(
            "Old XLS Actual Stock Total missing"
        )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Use the report's daily cell if it is present AND agrees
    # with Normative Stock / Days.
    #
    # Otherwise derive it.
    # --------------------------------------------------------

    derived_daily = (
        normative_stock
        / days
    )

    if daily is None:

        daily = derived_daily
        daily_method = "derived"

    else:

        error = (
            abs(
                daily
                - derived_daily
            )
            / max(
                abs(derived_daily),
                1
            )
        )

        if error <= 0.03:

            daily_method = (
                "report_cell_validated"
            )

        else:

            daily = derived_daily

            daily_method = (
                "derived_due_to_mismatch"
            )

    # --------------------------------------------------------
    # Actual Stock validation
    # --------------------------------------------------------

    actual_expected = None
    actual_error = None

    if (
        indigenous is not None
        and imported is not None
    ):

        actual_expected = (
            indigenous
            + imported
        )

        actual_error = (
            abs(
                actual_total
                - actual_expected
            )
            / max(
                abs(actual_expected),
                1
            )
        )

    return {

        "days": days,

        "daily": daily,

        "normative_stock": normative_stock,

        "actual_total": actual_total,

        "daily_col": daily_col,

        "normative_days_col": days_col,

        "normative_stock_col": normative_stock_col,

        "actual_total_col": actual_total_col,

        "indigenous_col": indigenous_col,

        "import_col": import_col,

        "days_method": daily_method,

        "actual_expected": actual_expected,

        "actual_error": actual_error,
    }


# ============================================================
# EXTRACT NEW XLS
# ============================================================

def extract_xls_new(
    book,
    grand_total_row
):

    # --------------------------------------------------------
    # Verified newer XLS layout:
    #
    # M = Normative Days
    # R = Normative Stock
    # S/U etc = Indigenous structure
    # V/X = Import structure
    # Y = Total
    #
    # The Daily Requirement Grand Total cell is not reliable.
    # Therefore derive:
    #
    # Daily = Normative Stock / Normative Days
    # --------------------------------------------------------

    days_col = 12
    normative_stock_col = 17

    indigenous_col = 18
    import_col = 21
    actual_total_col = 24

    days = num(
        book.get(
            grand_total_row,
            days_col
        )
    )

    normative_stock = num(
        book.get(
            grand_total_row,
            normative_stock_col
        )
    )

    indigenous = num(
        book.get(
            grand_total_row,
            indigenous_col
        )
    )

    imported = num(
        book.get(
            grand_total_row,
            import_col
        )
    )

    actual_total = num(
        book.get(
            grand_total_row,
            actual_total_col
        )
    )

    if days is None:

        raise RuntimeError(
            "New XLS Normative Days missing"
        )

    if normative_stock is None:

        raise RuntimeError(
            "New XLS Normative Stock missing"
        )

    if actual_total is None:

        raise RuntimeError(
            "New XLS Actual Stock Total missing"
        )

    if days <= 0:

        raise RuntimeError(
            f"Invalid Normative Days: {days}"
        )

    # --------------------------------------------------------
    # Daily Requirement is mathematically defined by:
    #
    # Normative Stock = Daily Requirement × Days
    # --------------------------------------------------------

    daily = (
        normative_stock
        / days
    )

    # --------------------------------------------------------
    # Actual validation
    # --------------------------------------------------------

    actual_expected = None
    actual_error = None

    if (
        indigenous is not None
        and imported is not None
    ):

        actual_expected = (
            indigenous
            + imported
        )

        actual_error = (
            abs(
                actual_total
                - actual_expected
            )
            / max(
                abs(actual_expected),
                1
            )
        )

    return {

        "days": days,

        "daily": daily,

        "normative_stock": normative_stock,

        "actual_total": actual_total,

        "daily_col": None,

        "normative_days_col": days_col,

        "normative_stock_col": normative_stock_col,

        "actual_total_col": actual_total_col,

        "indigenous_col": indigenous_col,

        "import_col": import_col,

        "days_method": "explicit",

        "actual_expected": actual_expected,

        "actual_error": actual_error,
    }


# ============================================================
# EXTRACT ONE FILE
# ============================================================

def extract_file(
    path
):

    book = load_workbook_data(
        path
    )

    grand_total_row = (
        find_grand_total_row(
            book
        )
    )

    layout = detect_layout(
        book,
        grand_total_row
    )

    # --------------------------------------------------------
    # Select extractor
    # --------------------------------------------------------

    if layout == "XLSX_STANDARD":

        data = extract_xlsx_standard(
            book,
            grand_total_row
        )

    elif layout == "XLS_OLD":

        data = extract_xls_old(
            book,
            grand_total_row
        )

    elif layout == "XLS_NEW":

        data = extract_xls_new(
            book,
            grand_total_row
        )

    else:

        raise RuntimeError(
            f"Unsupported layout: {layout}"
        )

    # ========================================================
    # VALIDATION
    # ========================================================

    daily = data["daily"]

    days = data["days"]

    normative_stock = (
        data["normative_stock"]
    )

    actual_total = (
        data["actual_total"]
    )

    # --------------------------------------------------------
    # Normative relationship
    # --------------------------------------------------------

    reconstructed_stock = (
        daily
        * days
    )

    normative_error = (
        abs(
            reconstructed_stock
            - normative_stock
        )
        / max(
            abs(normative_stock),
            1
        )
    )

    if normative_error > 0.03:

        raise RuntimeError(
            "Normative validation failed:\n"
            f"Days={days}\n"
            f"Daily={daily}\n"
            f"Normative Stock={normative_stock}\n"
            f"Calculated={reconstructed_stock}\n"
            f"Error={normative_error:.2%}"
        )

    # --------------------------------------------------------
    # Actual Stock validation
    # --------------------------------------------------------

    if (
        data["actual_error"]
        is not None
    ):

        if data["actual_error"] > 0.03:

            raise RuntimeError(
                "Actual Stock validation failed:\n"
                "Indigenous={data['actual_expected'] - "
                "num(book_get_safe(book, grand_total_row, data['import_col'])) "
                if False else ''
            )

    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    return {

        "Date": extract_date(
            path.name
        ),

        "Normative Stock Reqd. (Days)": (
            clean_num(days)
        ),

        "Daily Requirement @85% PLF (In '000 Tonnes)": (
            clean_num(daily)
        ),

        "Actual Stock - Total (In '000 Tonnes)": (
            clean_num(actual_total)
        ),

        # Audit
        "_File": path.name,

        "_Sheet": book.sheet_name,

        "_Layout": layout,

        "_Grand_Total_Row": (
            grand_total_row + 1
        ),

        "_Normative_Days_Column": (
            excel_col(
                data["normative_days_col"]
            )
            if data["normative_days_col"]
            is not None
            else None
        ),

        "_Daily_Column": (
            excel_col(
                data["daily_col"]
            )
            if data["daily_col"]
            is not None
            else "DERIVED"
        ),

        "_Normative_Stock_Column": (
            excel_col(
                data["normative_stock_col"]
            )
        ),

        "_Actual_Total_Column": (
            excel_col(
                data["actual_total_col"]
            )
        ),

        "_Indigenous_Column": (
            excel_col(
                data["indigenous_col"]
            )
        ),

        "_Import_Column": (
            excel_col(
                data["import_col"]
            )
        ),

        "_Days_Method": (
            data["days_method"]
        ),

        "_Normative_Validation_Error_Pct": (
            round(
                normative_error * 100,
                6
            )
        ),

        "_Actual_Expected_Total": (
            clean_num(
                data["actual_expected"]
            )
            if data["actual_expected"]
            is not None
            else None
        ),

        "_Actual_Validation_Error_Pct": (
            round(
                data["actual_error"] * 100,
                6
            )
            if data["actual_error"]
            is not None
            else None
        ),

        "_Status": "SUCCESS",
    }


# ============================================================
# PROCESS FILE
# ============================================================

def process_file(path):

    print(
        "\n" + "=" * 110
    )

    print(
        f"PROCESSING: {path.name}"
    )

    print(
        "=" * 110
    )

    result = extract_file(
        path
    )

    print(
        "\nEXTRACTED:"
    )

    print(
        "  Normative Days    :",
        result[
            "Normative Stock Reqd. (Days)"
        ]
    )

    print(
        "  Daily Requirement :",
        result[
            "Daily Requirement @85% PLF (In '000 Tonnes)"
        ]
    )

    print(
        "  Actual Stock Total:",
        result[
            "Actual Stock - Total (In '000 Tonnes)"
        ]
    )

    print(
        "\nSTRUCTURE:"
    )

    print(
        "  Layout            :",
        result["_Layout"]
    )

    print(
        "  Grand Total Row   :",
        result["_Grand_Total_Row"]
    )

    print(
        "  Normative Days    :",
        result["_Normative_Days_Column"]
    )

    print(
        "  Daily Requirement :",
        result["_Daily_Column"]
    )

    print(
        "  Normative Stock   :",
        result["_Normative_Stock_Column"]
    )

    print(
        "  Actual Total      :",
        result["_Actual_Total_Column"]
    )

    print(
        "\nVALIDATION:"
    )

    print(
        "  Normative error   :",
        result[
            "_Normative_Validation_Error_Pct"
        ],
        "%"
    )

    print(
        "  Actual error      :",
        result[
            "_Actual_Validation_Error_Pct"
        ],
        "%"
    )

    return result


# ============================================================
# DISCOVER FILES
# ============================================================

def discover_files():

    files = []

    files.extend(
        INPUT_DIR.glob(
            "dailyCoal1-*.xls"
        )
    )

    files.extend(
        INPUT_DIR.glob(
            "dailyCoal1-*.xlsx"
        )
    )

    return sorted(
        files,
        key=lambda p: extract_date(
            p.name
        )
    )


# ============================================================
# MAIN
# ============================================================

def main():

    files = discover_files()

    print(
        "=" * 110
    )

    print(
        f"Found {len(files)} files"
    )

    print(
        "=" * 110
    )

    results = []

    failures = []

    for path in files:

        try:

            result = process_file(
                path
            )

            results.append(
                result
            )

        except Exception as exc:

            print(
                "\n❌ FAILED:",
                path.name
            )

            print(
                "   ",
                str(exc)
            )

            failures.append(
                {
                    "File": path.name,
                    "Error": str(exc)
                }
            )

    # ========================================================
    # OUTPUT
    # ========================================================

    if results:

        output_rows = []

        audit_rows = []

        for result in results:

            output_rows.append(
                {
                    column: result[column]
                    for column in OUTPUT_COLUMNS
                }
            )

            audit_rows.append(
                result
            )

        result_df = pd.DataFrame(
            output_rows
        )

        audit_df = pd.DataFrame(
            audit_rows
        )

        result_df = (
            result_df
            .sort_values("Date")
            .drop_duplicates(
                "Date",
                keep="last"
            )
            .reset_index(
                drop=True
            )
        )

        result_df.to_csv(
            OUTPUT_CSV,
            index=False
        )

        audit_df.to_csv(
            AUDIT_CSV,
            index=False
        )

    else:

        result_df = pd.DataFrame(
            columns=OUTPUT_COLUMNS
        )

    # ========================================================
    # FINAL
    # ========================================================

    print(
        "\n" + "=" * 110
    )

    print(
        "FINAL RESULT"
    )

    print(
        "=" * 110
    )

    if result_df.empty:

        print(
            "❌ No successful files."
        )

    else:

        print(
            result_df.to_string(
                index=False
            )
        )

    print(
        "\nCSV:"
    )

    print(
        OUTPUT_CSV
    )

    print(
        "\nAUDIT:"
    )

    print(
        AUDIT_CSV
    )

    # ========================================================
    # FAILURES
    # ========================================================

    if failures:

        print(
            "\n" + "=" * 110
        )

        print(
            f"FAILED FILES: {len(failures)}"
        )

        print(
            "=" * 110
        )

        for failure in failures:

            print(
                "\n❌",
                failure["File"]
            )

            print(
                "   ",
                failure["Error"]
            )

    else:

        print(
            "\n" + "=" * 110
        )

        print(
            "✅ ALL FILES PROCESSED SUCCESSFULLY"
        )

        print(
            "=" * 110
        )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()

Found 27 files

PROCESSING: dailyCoal1-2026-08-12.xls

EXTRACTED:
  Normative Days    : 19.513902
  Daily Requirement : 3120.795401
  Actual Stock Total: 35766.3

STRUCTURE:
  Layout            : XLS_OLD
  Grand Total Row   : 270
  Normative Days    : L
  Daily Requirement : O
  Normative Stock   : R
  Actual Total      : Z

VALIDATION:
  Normative error   : 0.04402 %
  Actual error      : 0.001118 %

PROCESSING: dailyCoal1-2026-08-13.xls

EXTRACTED:
  Normative Days    : 19.513902
  Daily Requirement : 3120.795401
  Actual Stock Total: 35231.1

STRUCTURE:
  Layout            : XLS_OLD
  Grand Total Row   : 270
  Normative Days    : L
  Daily Requirement : O
  Normative Stock   : R
  Actual Total      : Z

VALIDATION:
  Normative error   : 0.04402 %
  Actual error      : 0.001419 %

PROCESSING: dailyCoal1-2026-08-14.xlsx

EXTRACTED:
  Normative Days    : 19.505582
  Daily Requirement : 3120.674355
  Actual Stock Total: 34894.3

STRUCTURE:
  Layout            : XLSX_STANDARD
  Grand Tot